# 05c · Multi-label Classification `[local]` (fine-tuning section: `[kaggle]`)

**Goal:** build a tagger that can say *"person + dog + sofa"* for one image: the right output layer, the right loss, threshold tuning, and the right metrics.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | "Which one?" vs. "which ones?" |
| 📐 Math | Sigmoid + binary cross-entropy, `pos_weight`, per-label thresholds, mAP and other multi-label metrics |
| 🔨 From scratch | BCE by hand, per-label threshold tuning (in `cvlib.metrics`, along with AP from 05b) |
| 🚀 Real run | PASCAL VOC 2007 (20 object types): linear probe on frozen features (Mac), full fine-tune (Kaggle) |
| 📊 Evaluate | Per-label AP vs. label frequency, F1 at 0.5 vs. tuned thresholds, exact match, prediction gallery |
| 🧪 Experiment | Softmax vs. sigmoid, `pos_weight`, crop strength |

**Run modes:** the linear-probe part runs fully on your Mac (~3 min). The fine-tuning section runs a 1-epoch smoke test locally and the real run on Kaggle.

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from cvlib.datasets import voc2007_multilabel, VOC_CLASSES, cache_images
from cvlib.loaders import GPULoader, random_resized_crop_flip, center_resize
from cvlib.metrics import multilabel_metrics, best_thresholds, average_precision
from cvlib import train as cvtrain

device = env.device
FULL = env.platform == "kaggle" or os.environ.get("CV_FULL") == "1"
AMP = device == "cuda"
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
print("fine-tuning mode:", "FULL" if FULL else "SMOKE (fine-tune numbers not meaningful; linear probe is real)")

---
# 🧠 1 · Intuition

**Single-label** asks *"which one?"*: the classes compete, exactly one wins. Softmax makes the probabilities sum to 1, so raising "dog" must lower "cat".

**Multi-label** asks *"which ones?"*: each label is an **independent yes/no question**. A photo can contain a person *and* a dog *and* a sofa, or none of the 20 object types. So:
- give each label its **own sigmoid** (a probability between 0 and 1, not shared with the others)
- train each with **binary cross-entropy**
- decide with a **threshold per label**, not argmax

Where it shows up: photo tagging, multiple defects on one part, several findings on one X-ray, document attributes (signed? stamped? handwritten?), content moderation.

---
# 📐 2 · The math

### Sigmoid + binary cross-entropy, per label
For logit $z_c$ and target $y_c \in \{0, 1\}$:

$$p_c = \sigma(z_c) = \frac{1}{1 + e^{-z_c}}, \qquad \text{BCE}_c = -\big[y_c \log p_c + (1 - y_c)\log(1 - p_c)\big]$$

The loss is the average over all labels and images. Each label is literally its own logistic regression sharing one backbone. Use `nn.BCEWithLogitsLoss` (it fuses the sigmoid in for numerical stability).

### Label imbalance: `pos_weight`
"person" appears in ~40% of VOC images, "sheep" in ~2%. For rare labels, "always say no" is right 98% of the time. `pos_weight` multiplies the positive term:

$$-\big[w_c\, y_c \log p_c + (1 - y_c)\log(1 - p_c)\big], \qquad w_c = \frac{\#\text{negatives}_c}{\#\text{positives}_c} \;\;(\text{often square-rooted to soften it})$$

### Initialize the bias to the label's frequency
A fresh head outputs logit 0, i.e. **p = 0.5 for every label**. But most labels are absent from ~95% of images, so the first stretch of training is spent just pushing ~19 of the 20 outputs down. Meanwhile the weights start fitting noise. Start each label's bias at its **log-odds** instead:

$$b_c = \log \frac{\pi_c}{1 - \pi_c}, \qquad \pi_c = \text{fraction of training images with label } c$$

Now the untrained model already predicts the base rate (e.g. p = 0.02 for "sheep"), and training only has to learn the *differences*. A one-line trick that's standard in detection heads (module 06) and any rare-label problem.

### Thresholds
0.5 is not magic. A label that's rare or hard often has its best F1 at a lower threshold. Choose **one threshold per label on validation data** (maximize F1, or hit a required precision/recall), then freeze them for test.

### Metrics

| Metric | Meaning | Threshold? |
|---|---|---|
| **AP per label, mAP** | ranking quality per label, averaged: the standard VOC/COCO metric | no |
| **macro-F1** | average F1 over labels: rare labels count equally | yes |
| **micro-F1** | F1 over all (image, label) decisions pooled: dominated by common labels | yes |
| **exact match** | fraction of images with *every* label right: very strict | yes |
| **Hamming loss** | fraction of wrong yes/no decisions: very lenient (most answers are "no") | yes |

---
# 3 · Data: PASCAL VOC 2007 as multi-label

VOC 2007 has bounding boxes for 20 object types (module 06 uses them for detection). For classification, an image's labels are simply *"which types appear at least once"*. We split the 5,011 *trainval* images into train / validation / test (70 / 15 / 15).

In [ ]:
paths, Y = voc2007_multilabel("trainval")
X_all, Y_all = cache_images("voc2007_trainval", paths, 256, labels=Y)
Y_all = Y_all.astype(np.float32)
rng = np.random.default_rng(0)
perm = rng.permutation(len(Y_all))
n_tr, n_va = int(0.7 * len(perm)), int(0.15 * len(perm))
idx_tr, idx_va, idx_te = np.sort(perm[:n_tr]), np.sort(perm[n_tr:n_tr + n_va]), np.sort(perm[n_tr + n_va:])
print(f"train {len(idx_tr)} | val {len(idx_va)} | test {len(idx_te)}")

freq = Y_all[idx_tr].mean(0)
n_labels = Y_all.sum(1)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [3, 1]})
order = np.argsort(-freq)
axes[0].bar([VOC_CLASSES[i] for i in order], freq[order]); axes[0].tick_params(axis="x", rotation=60)
axes[0].set_ylabel("fraction of images"); axes[0].set_title("How often each label appears (train)")
axes[1].hist(n_labels, bins=np.arange(0.5, 7.5, 1)); axes[1].set_xlabel("labels per image"); axes[1].set_title("Labels per image")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 6, figsize=(14, 2.9))
for ax, i in zip(axes, rng.choice(idx_tr, 6, replace=False)):
    ax.imshow(X_all[i]); ax.axis("off")
    ax.set_title(", ".join(VOC_CLASSES[c] for c in np.nonzero(Y_all[i])[0]), fontsize=8)
plt.tight_layout(); plt.show()

---
# 🔨 4 · From scratch

### 4.1 · BCE by hand vs. PyTorch, and why `BCEWithLogitsLoss`

In [ ]:
z = torch.tensor([[2.0, -1.0, 0.5], [-3.0, 4.0, 0.0]])
y = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0]])
p = torch.sigmoid(z)
bce_hand = -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()
print(f"by hand {bce_hand:.6f} | BCEWithLogitsLoss {F.binary_cross_entropy_with_logits(z, y):.6f}")

big = torch.tensor([[40.0]]); zero = torch.tensor([[0.0]])                  # a very confident wrong prediction
print("sigmoid then log  :", -torch.log(1 - torch.sigmoid(big)).item(), " ← float32 rounds σ(40) to exactly 1 → inf")
print("BCEWithLogitsLoss :", F.binary_cross_entropy_with_logits(big, zero).item(), " ← computed stably from the logit")

### 4.2 · Threshold tuning and AP

The AP function from 05b now lives in `cvlib.metrics`, alongside `multilabel_metrics` (all the metrics in the table above) and `best_thresholds` (per-label F1-maximizing threshold). Read `cvlib/metrics.py`: it's short.

---
# 🚀 5 · Approach A: linear probe on frozen features (runs on your Mac)

Extract ResNet-50 features once (05a's trick), then train 20 independent logistic regressions on them, one per label, all at once.

In [ ]:
backbone = timm.create_model("resnet50.a1_in1k", pretrained=True, num_classes=0).to(device).eval()

def loader(idx, train=False, scale=(0.5, 1.0), bs=64):
    aug = random_resized_crop_flip(224, scale=scale) if train else center_resize(224, 0.875)
    return GPULoader(np.asarray(X_all[idx]), Y_all[idx], batch_size=bs, shuffle=train, augment=aug, mean=MEAN, std=STD,
                     device=device, drop_last=train)

@torch.no_grad()
def features(idx):
    out = []
    for x, _ in loader(idx):
        with cvtrain.autocast(device, AMP):
            out.append(backbone.forward_head(backbone.forward_features(x), pre_logits=True).float().cpu())
    return torch.cat(out)

t0 = time.perf_counter()
F_tr, F_va, F_te = features(idx_tr), features(idx_va), features(idx_te)
Ytr, Yva, Yte = (torch.tensor(Y_all[i]) for i in (idx_tr, idx_va, idx_te))
print(f"features {tuple(F_tr.shape)} extracted in {time.perf_counter() - t0:.0f}s")

def prior_bias(Y):
    """Log-odds of each label's training frequency (see section 2)."""
    p = Y.float().mean(0).clamp(1e-3, 1 - 1e-3)
    return torch.log(p / (1 - p))

def train_head(Ftr, Ytr, pos_weight=None, loss="bce", epochs=300, lr=3e-3, wd=1e-4, init_prior=True):
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
    head = nn.Linear(Ftr.shape[1], Ytr.shape[1])
    if init_prior and loss == "bce":
        head.bias.data = prior_bias(Ytr)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        logits = head((Ftr - mu) / sd)
        if loss == "bce":
            l = F.binary_cross_entropy_with_logits(logits, Ytr, pos_weight=pos_weight)
        else:                                                               # "softmax": treat labels as a distribution
            l = -(Ytr / Ytr.sum(1, keepdim=True).clamp(min=1) * F.log_softmax(logits, 1)).sum(1).mean()
        opt.zero_grad(); l.backward(); opt.step()
    score = (lambda Fx: torch.sigmoid(head((Fx - mu) / sd))) if loss == "bce" else (lambda Fx: F.softmax(head((Fx - mu) / sd), 1))
    return lambda Fx: score(Fx).detach().numpy()

for init in (False, True):
    P = train_head(F_tr, Ytr, init_prior=init)(F_te)
    m = multilabel_metrics(P, Yte.numpy(), 0.5)
    print(f"bias init {'= label log-odds' if init else '= 0 (default)':<17}: mAP {m['mAP']:.3f} | macro-F1@0.5 {m['macro_f1']:.3f} | "
          f"Hamming loss {m['hamming_loss']:.3f} | mean predicted probability {P.mean():.3f} (true label rate {Ytr.mean():.3f})")

probe = train_head(F_tr, Ytr)
P_va, P_te = probe(F_va), probe(F_te)
m05 = multilabel_metrics(P_te, Yte.numpy(), 0.5)
print(f"\nlinear probe (prior init), test: mAP {m05['mAP']:.3f} | macro-F1@0.5 {m05['macro_f1']:.3f} | micro-F1@0.5 {m05['micro_f1']:.3f} | "
      f"exact match {m05['exact_match']:.3f} | Hamming loss {m05['hamming_loss']:.4f}")

---
# 📊 6 · Evaluate

### 6.1 · Per-label AP vs. how common the label is

In [ ]:
ap = m05["ap"]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(freq, ap)
for c in range(20):
    ax.annotate(VOC_CLASSES[c], (freq[c], ap[c]), fontsize=8, xytext=(3, 2), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel("label frequency in train (log)"); ax.set_ylabel("test AP")
ax.set_title(f"Per-label AP (mAP = {m05['mAP']:.3f})"); ax.grid(alpha=0.3); plt.show()
worst = np.argsort(ap)[:4]
print("hardest labels:", ", ".join(f"{VOC_CLASSES[c]} (AP {ap[c]:.2f})" for c in worst))

Common labels aren't automatically easy. Labels like *chair*, *bottle*, *pottedplant* and *diningtable* are usually small, partly hidden, or mere background in photos about something else, so they tend to have the lowest AP. Visually distinctive ones (*aeroplane*, *cat*, *train*) score highest even when rarer.

### 6.2 · Thresholds: 0.5 vs. tuned per label

Tune on **validation**, apply to **test**.

In [ ]:
thr = best_thresholds(P_va, Yva.numpy())
mt = multilabel_metrics(P_te, Yte.numpy(), thr)
print(f"{'':<22}{'macro-F1':>9}{'micro-F1':>10}{'exact match':>13}")
print(f"{'threshold 0.5':<22}{m05['macro_f1']:>9.3f}{m05['micro_f1']:>10.3f}{m05['exact_match']:>13.3f}")
print(f"{'tuned per label (val)':<22}{mt['macro_f1']:>9.3f}{mt['micro_f1']:>10.3f}{mt['exact_match']:>13.3f}")
print("\ntuned thresholds:", ", ".join(f"{VOC_CLASSES[c]} {thr[c]:.2f}" for c in np.argsort(thr)[:6]), "...")

Tuning moves each label's threshold to wherever its validation F1 peaks: **up** for labels the model over-predicts, **down** for labels it's timid about. That raises macro-F1 and exact match on test. mAP doesn't change, because it never used a threshold. **Report mAP to compare models; tune thresholds to deploy one.**

### 6.3 · What the tagger says

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6.8))
for ax, j in zip(axes.flat, rng.choice(len(idx_te), 10, replace=False)):
    i = idx_te[j]
    ax.imshow(X_all[i]); ax.axis("off")
    true_set = set(np.nonzero(Yte[j].numpy())[0]); pred_set = set(np.nonzero(P_te[j] >= thr)[0])
    lines = [f"✓ {VOC_CLASSES[c]}" for c in sorted(true_set & pred_set)] + [f"✗ missed {VOC_CLASSES[c]}" for c in sorted(true_set - pred_set)] \
          + [f"✗ extra {VOC_CLASSES[c]} ({P_te[j, c]:.2f})" for c in sorted(pred_set - true_set)]
    ax.set_title("\n".join(lines), fontsize=8, color="green" if true_set == pred_set else "darkred")
plt.tight_layout(); plt.show()

Typical errors: missing small background objects (a bottle on a table), and **co-occurrence guesses**, such as predicting *person* for a bicycle or horse photo because they usually appear together in training. Some "extra" predictions are actually right and missing from the labels. Real annotations are imperfect, and module 14 deals with that.

---
# 🚀 7 · Approach B: full fine-tuning `[kaggle]`

The backbone adapts too. Note the **milder crop** (scale 0.5–1.0 instead of 0.35–1.0): an aggressive random crop can cut a labeled object out of the image while its label stays "present", which teaches the model wrong answers. **Augmentations must preserve the labels.**

In [ ]:
def finetune(epochs, max_train=None, pos_weight=None, scale=(0.5, 1.0)):
    tr = idx_tr if max_train is None else idx_tr[:max_train]
    m = timm.create_model("resnet50.a1_in1k", pretrained=True, num_classes=20)
    m.fc.bias.data = prior_bias(torch.tensor(Y_all[tr]))              # start at each label's base rate
    head = list(m.fc.parameters()); body = [p for n, p in m.named_parameters() if not n.startswith("fc.")]
    opt = torch.optim.AdamW([{"params": body, "lr": 1e-4}, {"params": head, "lr": 1e-3}], weight_decay=0.05)
    tl = loader(tr, train=True, scale=scale, bs=32)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[1e-4, 1e-3], total_steps=epochs * len(tl), pct_start=0.1)
    pw = pos_weight.to(device) if pos_weight is not None else None
    cvtrain.fit(m, tl, loader(idx_va), nn.BCEWithLogitsLoss(pos_weight=pw), opt, epochs=epochs, device=device,
                batch_scheduler=sched, amp=AMP, verbose=FULL)
    def probs(idx):
        lg, _ = cvtrain.predict(m, loader(idx), device, amp=AMP)
        return torch.sigmoid(lg).numpy()
    return probs

t0 = time.perf_counter()
ft_probs = finetune(epochs=8 if FULL else 1, max_train=None if FULL else 256)
Pva_ft, Pte_ft = ft_probs(idx_va), ft_probs(idx_te)
thr_ft = best_thresholds(Pva_ft, Yva.numpy())
mft = multilabel_metrics(Pte_ft, Yte.numpy(), thr_ft)
print(f"fine-tuned ({'FULL' if FULL else 'SMOKE'}), test: mAP {mft['mAP']:.3f} | macro-F1 (tuned) {mft['macro_f1']:.3f} | {time.perf_counter() - t0:.0f}s")
print(f"linear probe,       test: mAP {m05['mAP']:.3f} | macro-F1 (tuned) {mt['macro_f1']:.3f}")

On Kaggle, full fine-tuning should beat the linear probe by several mAP points: the backbone learns VOC's object scales and contexts. In smoke mode (1 epoch on 256 images) it will be far worse. That run only checks the code.

---
# 🧪 8 · Experiments: predict first, then run (all on the fast linear probe)

**Experiment 1: softmax instead of sigmoid.** Train the head with a softmax over the 20 labels, as if exactly one were true. Predict the effect on images with several labels.

In [ ]:
# Prediction: ...
soft = train_head(F_tr, Ytr, loss="softmax")
P_soft = soft(F_te)
multi = Yte.numpy().sum(1) >= 2
for name, P in [("sigmoid + BCE", P_te), ("softmax", P_soft)]:
    mm = multilabel_metrics(P, Yte.numpy())
    second_label_ap = multilabel_metrics(P[multi], Yte.numpy()[multi])["mAP"]
    print(f"{name:<14}: mAP all images {mm['mAP']:.3f} | mAP on images with 2+ labels {second_label_ap:.3f}")
# Softmax makes labels compete: probability given to "person" is taken away from "horse", even when both are present.
# Its ranking can stay reasonable, but scores are no longer per-label yes/no probabilities you can threshold.

**Experiment 2: `pos_weight` for rare labels.** Predict how weighting positives (√ of negatives/positives) changes recall on rare labels at threshold 0.5.

In [ ]:
# Prediction: ...
pw = torch.sqrt((1 - Ytr.mean(0)) / Ytr.mean(0).clamp(min=1e-3))
weighted = train_head(F_tr, Ytr, pos_weight=pw)
P_w = weighted(F_te)
rare = np.argsort(freq)[:5]
for name, P in [("plain BCE", P_te), ("pos_weight", P_w)]:
    pred = P >= 0.5; T = Yte.numpy().astype(bool)
    rec = (pred & T).sum(0) / np.maximum(T.sum(0), 1)
    prec = (pred & T).sum(0) / np.maximum(pred.sum(0), 1)
    print(f"{name:<11}: rare-label recall {rec[rare].mean():.3f} | precision {prec[rare].mean():.3f} | mAP {multilabel_metrics(P, T)['mAP']:.3f}")
print("rare labels:", [VOC_CLASSES[c] for c in rare])
# Here pos_weight adds little, or even hurts slightly: prior-bias initialization already puts rare labels at the right
# base rate, and the ranking (mAP) is unchanged. In general pos_weight pushes rare-label probabilities up
# (more recall, less precision at a fixed threshold). It's one of several tools for the same problem
# (prior init, pos_weight, threshold tuning, more data). Keep it only if validation says it helps.

**Experiment 3: crop strength.** Evaluate the linear probe's features with a very tight crop (the central 50%) instead of the usual 87.5%. Predict which labels suffer most.

In [ ]:
# Prediction: ...
@torch.no_grad()
def features_crop(idx, frac):
    out = []
    for x, _ in GPULoader(np.asarray(X_all[idx]), Y_all[idx], batch_size=64, augment=center_resize(224, frac), mean=MEAN, std=STD, device=device):
        out.append(backbone.forward_head(backbone.forward_features(x), pre_logits=True).float().cpu())
    return torch.cat(out)

P_tight = probe(features_crop(idx_te, 0.5))
ap_tight = multilabel_metrics(P_tight, Yte.numpy())["ap"]
drop = ap - ap_tight
print(f"mAP: crop 87.5% → {m05['mAP']:.3f} | crop 50% → {np.nanmean(ap_tight):.3f}")
print("biggest AP drops:", ", ".join(f"{VOC_CLASSES[c]} ({drop[c]:+.2f})" for c in np.argsort(-drop)[:5]))
# Objects near the image edges get cut out while their label still says "present": the same reason
# training crops must be milder for multi-label data than for single-label data.

---
# 📝 9 · Recap

**Cheat sheet**
- Multi-label = one **independent sigmoid** per label + **binary cross-entropy** (`BCEWithLogitsLoss`), no softmax.
- Decide with **per-label thresholds** tuned on validation, not argmax and not blindly 0.5.
- **mAP** (threshold-free) to compare models; **macro-F1** if every label matters; **micro-F1** for overall decisions; **exact match** is strict, **Hamming loss** lenient.
- Rare labels: `pos_weight`, threshold tuning, more data. Watch for **co-occurrence** shortcuts.
- Augmentations must keep labels true: use **milder crops** when objects can be cut out.
- Linear probe on frozen features = a strong, minutes-cheap baseline; fine-tuning adds several mAP points.

**Interview-style questions**
1. Why use sigmoid + BCE instead of softmax + cross-entropy for multi-label problems?
2. How would you choose decision thresholds for a 20-label tagger? On which data?
3. mAP vs. macro-F1 vs. micro-F1: what does each reward, and which would you report to compare two models?
4. Your tagger misses rare labels. List three fixes.
5. Why can random cropping be harmful for multi-label training?
6. Exact-match ratio is 0.45 but Hamming loss is 0.03. Is the model good? Explain both numbers.

---
# 🎉 Phase 2 complete

You've built CNNs from the convolution up (03), know the architectures and how to benchmark them (04), and can adapt pretrained models to new data, train them well, and evaluate them properly (05).

**Next:** Phase 3, *seeing where*: object detection (06) and segmentation (07).